In [7]:
from pyspark.sql import functions as F
from multitudcsd.config import get_spark_session
from multitudcsd.storage import read_delta

In [13]:
spark = get_spark_session("diagnostico-ml")
gold = read_delta(spark, "gold", "gold_line_reliability")


In [14]:
#tamaño total
print("filas:", gold.count())
print("lineas distintas:", gold.select("route_id").distinct().count())

filas: 7141
lineas distintas: 936


In [15]:
print(gold.columns)

['route_id', 'service_date', 'hour_of_day', 'avg_delay_seconds', 'pct_on_time', 'num_updates']


In [16]:
#dias y horas
gold.groupBy("service_date").agg(
    F.countDistinct("hour_of_day").alias("horas"),
    F.count("*").alias("filas"),
).orderBy("service_date").show(50)

+------------+-----+-----+
|service_date|horas|filas|
+------------+-----+-----+
|  2026-08-23|    1|  234|
|  2026-08-27|    1|  189|
|  2026-09-02|    1|  127|
|  2026-09-04|    2| 1010|
|  2026-09-08|    8| 5581|
+------------+-----+-----+



In [ ]:
#franjas horarias total
franjas = gold.select("service_date", "hour_of_day").distinct().count()
print("franjas distintas:", franjas)

In [ ]:
gold.select(
    F.min("avg_delay_seconds"), F.max("avg_delay_seconds"), F.avg("avg_delay_seconds")
).show()

In [18]:
read_delta(spark, "gold", "gold_delay_predictions") \
    .orderBy(F.desc("predicted_delay_seconds")).show(10, truncate=False)

+---------+-------------------+-----------+-----------------------+---------------------------+--------------------------+------+
|route_id |predicted_for_ts   |hour_of_day|predicted_delay_seconds|last_observed_delay_seconds|predicted_at              |source|
+---------+-------------------+-----------+-----------------------+---------------------------+--------------------------+------+
|5256_3   |2026-09-08 16:00:00|16         |13124.32409527645      |15940.666666666666         |2026-09-10 03:16:30.993826|real  |
|5359_700 |2026-09-08 17:00:00|17         |1782.3691602918216     |3302.0                     |2026-09-10 03:16:30.993826|real  |
|5235_700 |2026-09-08 21:00:00|21         |1678.1216945824444     |2613.964285714286          |2026-09-10 03:16:30.993826|real  |
|27274_700|2026-09-08 18:00:00|18         |1421.3082545394968     |1757.837837837838          |2026-09-10 03:16:30.993826|real  |
|26780_700|2026-09-08 21:00:00|21         |1251.1008147531684     |1629.89010989011       